# Lesson 01 companion — runnable Temporal tour.

> This notebook bootstraps an in-process Temporal server via `WorkflowEnvironment.start_local()` so you can poke the primitives without needing the docker stack up. The "real" workflow for every later lesson runs against `make temporal-up` instead.
> Read [`README.md`](./README.md) in this folder alongside.

Four primitives: **workflow**, **activity**, **worker**, **task queue**.
This notebook touches every one.

In [16]:
from datetime import timedelta

# Standard temporalio imports we'll use throughout the track.
from temporalio import activity, workflow
from temporalio.testing import WorkflowEnvironment
from temporalio.worker import UnsandboxedWorkflowRunner, Worker

## 1. Define an activity

Activities are where side effects live: HTTP calls, file I/O, model
inference. The decorator marks the function as something a workflow is
allowed to invoke through Temporal (instead of calling directly).

In [17]:
@activity.defn
async def say_hello(name: str) -> str:
    """Side-effecting function. Could call an API; here it just returns a string."""
    return f"Hello, {name}!"

## 2. Define a workflow

Workflow code is **deterministic** — no `random`, no `datetime.now()`, no
`httpx.get()`. Anything non-deterministic goes in an activity, which the
workflow calls via `workflow.execute_activity(...)`. The activity result
is memoized in workflow history so replay reproduces the same state.

In [18]:
@workflow.defn
class GreetWorkflow:
    """The simplest possible Temporal workflow — one activity call."""

    @workflow.run
    async def run(self, name: str) -> str:
        return await workflow.execute_activity(
            say_hello,
            name,
            start_to_close_timeout=timedelta(seconds=10),
        )

## 3. Start an ephemeral server + worker

`WorkflowEnvironment.start_local()` spins up a real Temporal server in a
subprocess and tears it down when the `async with` exits. Use it for
notebooks and tests; use the docker stack (`make temporal-up`) for "real"
multi-terminal lessons.

In [19]:
async def run_demo() -> str:
    async with await WorkflowEnvironment.start_local() as env:
        # The worker polls a task queue and runs whatever code it finds.
        # `UnsandboxedWorkflowRunner` is used here because Jupyter's `__main__`
        # module lacks a `__file__` attribute that the default sandbox needs.
        # In a regular .py file (every other lesson) you omit this and let the
        # sandbox enforce determinism for you.
        async with Worker(
            env.client,
            task_queue="lesson-01-tour",
            workflows=[GreetWorkflow],
            activities=[say_hello],
            workflow_runner=UnsandboxedWorkflowRunner(),
        ):
            # Start the workflow and await its result.
            result: str = await env.client.execute_workflow(
                GreetWorkflow.run,
                "Henry",
                id="lesson-01-greet",
                task_queue="lesson-01-tour",
            )
            return result

In [20]:
result = await run_demo()  # type: ignore[top-level-await]  # ok in Jupyter
print(result)

2026-05-25T02:05:18.104968Z  WARN temporalio_sdk_core::worker: shutdown_worker rpc errored during worker shutdown: Status { code: Unknown, message: "transport error", source: Some(tonic::transport::Error(Transport, hyper::Error(Io, Custom { kind: BrokenPipe, error: "stream closed because of a broken pipe" }))) }
Hello, Henry!


## 4. What just happened?

1. The client called `execute_workflow(GreetWorkflow.run, "Henry", ...)`.
2. Temporal recorded `WorkflowExecutionStarted` in workflow history.
3. The worker picked up the workflow task, ran `GreetWorkflow.run` until
   it hit `await workflow.execute_activity(say_hello, ...)`.
4. Temporal scheduled an activity task. The worker picked it up.
   `say_hello` ran. The result was sent back, recorded as
   `ActivityTaskCompleted`, and returned to the workflow.
5. The workflow returned. Temporal recorded `WorkflowExecutionCompleted`.

If the worker had crashed at step 4, Temporal would re-schedule the
activity. If it crashed during the workflow body, replay reconstructs
state from history.

## 5. (Optional) Connect to your docker stack instead

Once `make temporal-up` is running, you can connect via
`learn_pydantic_ai.connect()` and start the same workflow there. Then it
shows up in the Temporal UI at <http://localhost:8080>.

In [ ]:
# Run against your local docker stack (requires `make temporal-up`).

import uuid

from learn_pydantic_ai import connect

client = await connect()
# Same `UnsandboxedWorkflowRunner` trick as the ephemeral cell — Jupyter's
# `__main__` has no `__file__`, and the default SandboxedWorkflowRunner
# needs that to re-import the workflow module during validation. Skip it
# and you get: `AttributeError: module '__main__' has no attribute '__file__'`.
async with Worker(
    client,
    task_queue="lesson-01-tour",  # any task queue name; clients/workers must agree
    workflows=[GreetWorkflow],
    activities=[say_hello],
    workflow_runner=UnsandboxedWorkflowRunner(),
):
    # uuid suffix so the docker stack (which persists workflow IDs) doesn't
    # reject a re-run with WorkflowAlreadyStartedError.
    result = await client.execute_workflow(
        GreetWorkflow.run,
        "Henry",
        id=f"lesson-01-greet-docker-{uuid.uuid4().hex[:8]}",
        task_queue="lesson-01-tour",
    )
    print(result)

## What's next

Lesson 02 wraps a real pydantic-ai `Agent` in a workflow. The `say_hello`
activity becomes an LLM call; the boilerplate stays the same.